Import libraries

In [1]:
import duckdb
import os
import pandas as pd

Connect to DuckDB

In [2]:
conn = duckdb.connect(
    "../customer_support.duckdb",
    read_only=True
)

print("Connected successfully")

Connected successfully


Helper function

In [3]:
def run_query(query):
    """
    Execute SQL query against DuckDB
    and return results as a pandas dataframe.
    """
    
    return conn.execute(query).df()

In [4]:
df = run_query("""
SELECT *
FROM main.fact_ticket_metrics
LIMIT 5
""")

df

,ticket_id,customer_id,product_id,ticket_type,ticket_status,ticket_priority,ticket_channel,product_name,purchase_date,first_response_time,time_to_resolution,cleaned_first_response_time,cleaned_resolution_time,response_time_quality_flag,resolution_hours,first_response_hours,customer_satisfaction_rating
0,1,bd76cedb7edb7edad02693111b3f9a12,f879ad7bfea7006aa52bba6f4b8d96d9,Technical issue,Pending Customer Response,Critical,Social media,GoPro Hero,2021-03-22,2023-06-01 12:15:36,NaT,2023-06-01 12:15:36,NaT,Valid,<NA>,19236,NaN
1,2,48c46c02eff74a80d352e0154fd3a18b,62459136cefea3f345f166ae91b2b137,Technical issue,Pending Customer Response,Critical,Chat,LG Smart TV,2021-05-22,2023-06-01 16:45:38,NaT,2023-06-01 16:45:38,NaT,Valid,<NA>,17776,NaN
2,3,36045d9133e6cbb83b69f86e55de8c38,4b01fecc9aaea14fd9d81c1639df377e,Technical issue,Closed,Low,Social media,Dell XPS,2020-07-14,2023-06-01 11:14:38,2023-06-01 18:05:38,2023-06-01 11:14:38,2023-06-01 18:05:38,Valid,7,25259,3.0
3,4,d0ad5318f2595dd7e537ba738a4b743c,2482944e6048d8f3b76417f0f5af9074,Billing inquiry,Closed,Low,Social media,Microsoft Office,2020-11-13,2023-06-01 07:29:40,2023-06-01 01:57:40,2023-06-01 01:57:40,2023-06-01 07:29:40,Corrected,6,22321,3.0
4,5,8f6418b4d3a5c97d9b32a75060b9aa3a,d59bcdb3e60ada2624c0becf9d391e7d,Billing inquiry,Closed,Low,Email,Autodesk AutoCAD,2020-02-04,2023-06-01 00:12:42,2023-06-01 19:53:42,2023-06-01 00:12:42,2023-06-01 19:53:42,Valid,19,29112,1.0


Create export folder automatically

In [5]:
export_path = "../exports"

os.makedirs(
    export_path,
    exist_ok=True
)

print("Export folder ready")

Export folder ready


Export fact table (fact_ticket_metrics)

In [6]:
run_query("""
COPY main.fact_ticket_metrics
TO '../exports/fact_ticket_metrics.parquet'
(FORMAT PARQUET);
""")

,Count
0,8791


Export customer dimension

In [7]:
run_query("""
COPY main.dim_customers
TO '../exports/dim_customers.parquet'
(FORMAT PARQUET);
""")

,Count
0,8469


Export products dimension

In [8]:
run_query("""
COPY main.dim_products
TO '../exports/dim_products.parquet'
(FORMAT PARQUET);
""")

,Count
0,42


Verify exports

In [9]:
os.listdir("../exports")

['dim_customers.parquet',
 'dim_products.parquet',
 'fact_ticket_metrics.parquet']

Validate Parquet files

In [ ]:
fact_check = run_query("""
SELECT *
FROM read_parquet(
'../exports/fact_ticket_metrics.parquet'
)
LIMIT 5
""")

fact_check

,ticket_id,customer_id,product_id,ticket_type,ticket_status,ticket_priority,ticket_channel,product_name,purchase_date,first_response_time,time_to_resolution,cleaned_first_response_time,cleaned_resolution_time,response_time_quality_flag,resolution_hours,first_response_hours,customer_satisfaction_rating
0,1,bd76cedb7edb7edad02693111b3f9a12,f879ad7bfea7006aa52bba6f4b8d96d9,Technical issue,Pending Customer Response,Critical,Social media,GoPro Hero,2021-03-22,2023-06-01 12:15:36,NaT,2023-06-01 12:15:36,NaT,Valid,<NA>,19236,NaN
1,2,48c46c02eff74a80d352e0154fd3a18b,62459136cefea3f345f166ae91b2b137,Technical issue,Pending Customer Response,Critical,Chat,LG Smart TV,2021-05-22,2023-06-01 16:45:38,NaT,2023-06-01 16:45:38,NaT,Valid,<NA>,17776,NaN
2,3,36045d9133e6cbb83b69f86e55de8c38,4b01fecc9aaea14fd9d81c1639df377e,Technical issue,Closed,Low,Social media,Dell XPS,2020-07-14,2023-06-01 11:14:38,2023-06-01 18:05:38,2023-06-01 11:14:38,2023-06-01 18:05:38,Valid,7,25259,3.0
3,4,d0ad5318f2595dd7e537ba738a4b743c,2482944e6048d8f3b76417f0f5af9074,Billing inquiry,Closed,Low,Social media,Microsoft Office,2020-11-13,2023-06-01 07:29:40,2023-06-01 01:57:40,2023-06-01 01:57:40,2023-06-01 07:29:40,Corrected,6,22321,3.0
4,5,8f6418b4d3a5c97d9b32a75060b9aa3a,d59bcdb3e60ada2624c0becf9d391e7d,Billing inquiry,Closed,Low,Email,Autodesk AutoCAD,2020-02-04,2023-06-01 00:12:42,2023-06-01 19:53:42,2023-06-01 00:12:42,2023-06-01 19:53:42,Valid,19,29112,1.0


: 